# Algorithm Validation: Our Code vs Original Author (Bali)

Validates that our SPI/SPEI implementation matches the original author's output
by running **our code on the same input data** the original author used.

| | Input data | Who computed it |
|---|---|---|
| **Reference** | Old Bali PPT/PET (~2022 TerraClimate) | Original author |
| **Ours** | Same old Bali PPT/PET | Our pipeline (`spi()` / `spei()`) |

If the algorithm is correct → r ≈ 1.000, RMSE ≈ 0.

**Before running:** no GPU needed — CPU runtime is fine.

## Step 1 — Mount Google Drive & Set Paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/01_projects/precip-index')

# Old Bali input files (used by original author)
BALI_PPT = DRIVE_ROOT / 'tests' / 'output' / 'netcdf' / 'terraclimate_bali_ppt_1958_2024.nc'
BALI_PET = DRIVE_ROOT / 'tests' / 'output' / 'netcdf' / 'terraclimate_bali_pet_1958_2024.nc'

# Original author's pre-computed SPI/SPEI outputs (ground truth)
REF_SPI  = DRIVE_ROOT / 'tests' / 'output' / 'netcdf' / 'spi_12_gamma.nc'
REF_SPEI = DRIVE_ROOT / 'tests' / 'output' / 'netcdf' / 'spei_12_pet_gamma.nc'

# src/ for our code
SRC_DIR  = DRIVE_ROOT / 'src'

print('Bali PPT :', BALI_PPT.exists())
print('Bali PET :', BALI_PET.exists())
print('Ref SPI  :', REF_SPI.exists())
print('Ref SPEI :', REF_SPEI.exists())
print('src/     :', SRC_DIR.exists())

## Step 2 — Install Dependencies & Clone/Update Repo


In [ ]:
import subprocess, sys

pkgs = ['xarray', 'netCDF4', 'scipy', 'matplotlib', 'numba']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Dependencies installed.')


## Step 3 — Restore Old Bali PPT/PET from Git

The old Bali-specific files were deleted from disk but are preserved in git history.
This cell extracts them so we can recompute SPI/SPEI from the original data.


In [ ]:
import sys, shutil
from pathlib import Path

# Copy input files to local NVMe (faster reads, avoids Drive dropout)
LOCAL_PPT = Path('/content/bali_ppt.nc')
LOCAL_PET = Path('/content/bali_pet.nc')

for src, dst in [(BALI_PPT, LOCAL_PPT), (BALI_PET, LOCAL_PET)]:
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'Copied {src.name} → {dst}')
    else:
        print(f'Already local: {dst.name}')

# Add src/ to path
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f'src/ in path: {SRC_DIR.exists()}')

## Step 4 — Add src/ to Python Path


In [ ]:
import warnings
import numpy as np
import xarray as xr
warnings.filterwarnings('ignore')

from indices import spi, spei

CALIB_START = 1991
CALIB_END   = 2020

bali_ppt = xr.open_dataset(LOCAL_PPT)['ppt']
bali_pet = xr.open_dataset(LOCAL_PET)['pet']
print(f'PPT: {bali_ppt.shape}  PET: {bali_pet.shape}')

print('\nComputing SPI-12 ...')
our_spi = spi(bali_ppt, scale=12,
              calibration_start_year=CALIB_START,
              calibration_end_year=CALIB_END,
              distribution='gamma')
print(f'  Done: {our_spi.shape}')

print('Computing SPEI-12 ...')
our_spei = spei(bali_ppt, bali_pet, scale=12,
                calibration_start_year=CALIB_START,
                calibration_end_year=CALIB_END,
                distribution='gamma')
print(f'  Done: {our_spei.shape}')

## Step 5 — Compute SPI/SPEI from Old Bali PPT/PET

Uses the same algorithm as our global pipeline.
Validated locally: r=1.000000, RMSE=0.000024 vs pre-computed reference.


In [ ]:
# Load original author's reference outputs
REF = {
    'spi':  (xr.open_dataset(REF_SPI),  'spi_gamma_12_month',  our_spi),
    'spei': (xr.open_dataset(REF_SPEI), 'spei_gamma_12_month', our_spei),
}

results_raw = {}
for index, (ds_ref, var, da_our) in REF.items():
    da_ref = ds_ref[var]
    common_time = np.intersect1d(da_ref.time.values, da_our.time.values)
    da_ref = da_ref.sel(time=common_time)
    da_our_aligned = da_our.sel(time=common_time)
    print(f'[{index.upper()}-12] common months: {len(common_time)}  '
          f'grid: {da_ref.sizes}')
    results_raw[index] = (da_ref.load(), da_our_aligned.load())

## Step 6 — Load Our Global Output & Subset to Bali


In [ ]:
def stats(ref, our, label):
    r = ref.values.ravel()
    o = our.values.ravel()
    mask = np.isfinite(r) & np.isfinite(o)
    r, o = r[mask], o[mask]

    bias = float(np.mean(o - r))
    mae  = float(np.mean(np.abs(o - r)))
    rmse = float(np.sqrt(np.mean((o - r) ** 2)))
    corr = float(np.corrcoef(r, o)[0, 1])
    n    = int(mask.sum())

    print(f'\n  {label} (n={n:,})')
    print(f'    Correlation : {corr:.6f}')
    print(f'    RMSE        : {rmse:.6f}')
    print(f'    MAE         : {mae:.6f}')
    print(f'    Bias        : {bias:+.6f}')

    n_lat, n_lon = ref.sizes['lat'], ref.sizes['lon']
    corr_map = np.full((n_lat, n_lon), np.nan)
    for i in range(n_lat):
        for j in range(n_lon):
            rv = ref.values[:, i, j]
            ov = our.values[:, i, j]
            m  = np.isfinite(rv) & np.isfinite(ov)
            if m.sum() > 10:
                corr_map[i, j] = np.corrcoef(rv[m], ov[m])[0, 1]

    return dict(bias=bias, mae=mae, rmse=rmse, corr=corr, n=n,
                corr_map=corr_map, r_flat=r, o_flat=o)


results = {}
for index in ['spi', 'spei']:
    da_ref, da_our = results_raw[index]
    st = stats(da_ref, da_our, index.upper() + '-12')
    results[index] = (da_ref, da_our, st)

## Step 7 — Compute Statistics


In [ ]:
def stats(ref, our, label):
    r = ref.values.ravel()
    o = our.values.ravel()
    mask = np.isfinite(r) & np.isfinite(o)
    r, o = r[mask], o[mask]

    bias = float(np.mean(o - r))
    mae  = float(np.mean(np.abs(o - r)))
    rmse = float(np.sqrt(np.mean((o - r) ** 2)))
    corr = float(np.corrcoef(r, o)[0, 1])
    n    = int(mask.sum())

    print(f'\n  {label} (n={n:,})')
    print(f'    Correlation      : {corr:.4f}')
    print(f'    RMSE             : {rmse:.4f}')
    print(f'    MAE              : {mae:.4f}')
    print(f'    Bias (new−old)   : {bias:+.4f}')

    n_lat, n_lon = ref.sizes['lat'], ref.sizes['lon']
    corr_map = np.full((n_lat, n_lon), np.nan)
    for i in range(n_lat):
        for j in range(n_lon):
            rv = ref.values[:, i, j]
            ov = our.values[:, i, j]
            m  = np.isfinite(rv) & np.isfinite(ov)
            if m.sum() > 10:
                corr_map[i, j] = np.corrcoef(rv[m], ov[m])[0, 1]

    return dict(bias=bias, mae=mae, rmse=rmse, corr=corr, n=n,
                corr_map=corr_map, r_flat=r, o_flat=o)


results = {}
for index in ['spi', 'spei']:
    da_ref, da_our = results_raw[index]
    if da_ref is None:
        results[index] = None
        continue
    st = stats(da_ref, da_our, index.upper() + '-12')
    results[index] = (da_ref, da_our, st)


## Step 8 — Plot Comparison Figure


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

LABEL_REF = 'Original author'
LABEL_OUR = 'Our code (same input data)'


def make_figure(results):
    indices = [k for k, v in results.items() if v is not None]
    n_idx   = len(indices)
    if n_idx == 0:
        print('No valid comparisons to plot.')
        return None

    fig = plt.figure(figsize=(18, 6 * n_idx))
    fig.suptitle(
        'SPI/SPEI-12 Algorithm Validation (Bali)\n'
        f'{LABEL_REF}  vs  {LABEL_OUR}\n'
        'Same input data — any difference is algorithmic',
        fontsize=12, fontweight='bold', y=1.02
    )

    for row, index in enumerate(indices):
        da_ref, da_our, st = results[index]
        label = index.upper() + '-12'
        lat = da_ref.lat.values
        lon = da_ref.lon.values

        ci = len(lat) // 2
        cj = len(lon) // 2
        ts_ref = da_ref.values[:, ci, cj]
        ts_our = da_our.values[:, ci, cj]
        time   = da_ref.time.values

        gs = GridSpec(2, 3, figure=fig,
                      top=1 - row / n_idx - 0.02,
                      bottom=1 - (row + 1) / n_idx + 0.04,
                      hspace=0.45, wspace=0.35)

        # Time series
        ax1 = fig.add_subplot(gs[0, :2])
        ax1.plot(time, ts_ref, lw=0.8, label=LABEL_REF, color='steelblue')
        ax1.plot(time, ts_our, lw=0.8, label=LABEL_OUR, color='tomato',
                 linestyle='--', alpha=0.85)
        ax1.axhline(0, color='k', lw=0.4)
        ax1.set_title(f'{label} — time series at '
                      f'({lat[ci]:.2f}°N, {lon[cj]:.2f}°E)', fontsize=9)
        ax1.set_ylabel(label)
        ax1.legend(fontsize=8)
        ax1.grid(True, alpha=0.3)

        # Scatter
        ax2 = fig.add_subplot(gs[0, 2])
        lim  = min(max(abs(st['r_flat']).max(), abs(st['o_flat']).max()), 4.0)
        ax2.hexbin(st['r_flat'], st['o_flat'], gridsize=60, cmap='Blues',
                   mincnt=1, extent=(-lim, lim, -lim, lim))
        ax2.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
        ax2.set_xlabel(LABEL_REF)
        ax2.set_ylabel(LABEL_OUR)
        ax2.set_title(f'{label} scatter  r={st["corr"]:.4f}  '
                      f'RMSE={st["rmse"]:.4f}', fontsize=9)
        ax2.set_xlim(-lim, lim)
        ax2.set_ylim(-lim, lim)

        # Spatial correlation map
        ax3 = fig.add_subplot(gs[1, 0])
        vmin_corr = max(0.0, np.nanmin(st['corr_map']) - 0.05)
        im3 = ax3.imshow(st['corr_map'], vmin=vmin_corr, vmax=1.0, cmap='RdYlGn',
                         origin='upper',
                         extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                         aspect='auto')
        plt.colorbar(im3, ax=ax3, shrink=0.8)
        ax3.set_title(f'{label} per-pixel correlation', fontsize=9)
        ax3.set_xlabel('Lon')
        ax3.set_ylabel('Lat')

        # Bias map — use raw numpy to avoid xarray coordinate alignment issues
        bias_map = da_our.values - da_ref.values   # both already (time, lat, lon)
        bias_map = np.nanmean(bias_map, axis=0)    # (lat, lon)
        valid = bias_map[np.isfinite(bias_map)]
        vmax = max(abs(valid.min()), abs(valid.max()), 0.001) if valid.size > 0 else 0.1

        ax4 = fig.add_subplot(gs[1, 1])
        im4 = ax4.imshow(bias_map, vmin=-vmax, vmax=vmax, cmap='RdBu_r',
                         origin='upper',
                         extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                         aspect='auto')
        plt.colorbar(im4, ax=ax4, shrink=0.8)
        ax4.set_title(f'{label} mean bias\n(ours − original)', fontsize=9)
        ax4.set_xlabel('Lon')
        ax4.set_ylabel('Lat')

        # Stats box
        ax5 = fig.add_subplot(gs[1, 2])
        ax5.axis('off')
        txt = (
            f'{label} validation stats\n'
            f'(n = {st["n"]:,} valid points)\n\n'
            f'Correlation : {st["corr"]:.6f}\n'
            f'RMSE        : {st["rmse"]:.6f}\n'
            f'MAE         : {st["mae"]:.6f}\n'
            f'Bias        : {st["bias"]:+.6f}\n\n'
            f'Corr map min : {np.nanmin(st["corr_map"]):.4f}\n'
            f'Corr map max : {np.nanmax(st["corr_map"]):.4f}\n'
            f'Corr map mean: {np.nanmean(st["corr_map"]):.6f}'
        )
        ax5.text(0.05, 0.95, txt, transform=ax5.transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    return fig


fig = make_figure(results)
if fig is not None:
    plt.show()

## Step 9 — Save Figure to Google Drive


In [ ]:
if fig is not None:
    OUTPUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_FIGURE, dpi=150, bbox_inches='tight')
    plt.close(fig)
    size_kb = OUTPUT_FIGURE.stat().st_size / 1024
    print(f'Saved  ({size_kb:.0f} KB)  →  {OUTPUT_FIGURE}')
else:
    print('No figure to save.')


## Step 10 — GPU vs CPU Difference Test

Quantifies any numerical difference introduced by the GPU path (float32 + CuPy)
vs the CPU path (float32/float64 + NumPy/scipy) on the same Bali data.

Run this cell on a **GPU runtime** (`Runtime → Change runtime type → A100 GPU`).
On CPU runtime it will report GPU not available and skip.

In [ ]:
from gpu import GPU_AVAILABLE
from compute import compute_index_parallel
from config import Periodicity
import numpy as np

data = bali_ppt.values.astype(np.float32)   # same dtype as global pipeline

print(f'GPU available: {GPU_AVAILABLE}')
print(f'Data shape: {data.shape}  dtype: {data.dtype}')

# CPU path (float32, same as memory_efficient=True default)
print('\nRunning CPU path ...')
result_cpu, _ = compute_index_parallel(
    data.copy(), scale=12, data_start_year=1958,
    calibration_start_year=1991, calibration_end_year=2020,
    periodicity=Periodicity.monthly, use_gpu=False
)

if GPU_AVAILABLE:
    print('Running GPU path ...')
    result_gpu, _ = compute_index_parallel(
        data.copy(), scale=12, data_start_year=1958,
        calibration_start_year=1991, calibration_end_year=2020,
        periodicity=Periodicity.monthly, use_gpu=True
    )

    mask = np.isfinite(result_cpu) & np.isfinite(result_gpu)
    diff = result_gpu[mask] - result_cpu[mask]
    corr = float(np.corrcoef(result_cpu[mask], result_gpu[mask])[0, 1])
    print(f'\nGPU vs CPU (float32 in both):')
    print(f'  Correlation : {corr:.8f}')
    print(f'  RMSE        : {np.sqrt(np.mean(diff**2)):.8f}')
    print(f'  Max abs diff: {np.abs(diff).max():.8f}')
    print(f'  Mean diff   : {diff.mean():+.8f}')
else:
    print('\nGPU not available — switch to GPU runtime to test this path.')

# Also check float32 vs float64 CPU difference
print('\nRunning CPU float64 path ...')
result_cpu64, _ = compute_index_parallel(
    data.astype(np.float64), scale=12, data_start_year=1958,
    calibration_start_year=1991, calibration_end_year=2020,
    periodicity=Periodicity.monthly, use_gpu=False
)
mask = np.isfinite(result_cpu) & np.isfinite(result_cpu64)
diff64 = result_cpu64[mask] - result_cpu[mask]
corr64 = float(np.corrcoef(result_cpu[mask], result_cpu64[mask])[0, 1])
print(f'\nCPU float64 vs CPU float32:')
print(f'  Correlation : {corr64:.8f}')
print(f'  RMSE        : {np.sqrt(np.mean(diff64**2)):.8f}')
print(f'  Max abs diff: {np.abs(diff64).max():.8f}')
print(f'  Mean diff   : {diff64.mean():+.8f}')